# Scenarios and stress testing

**Prerequisites:** complete `01_foundations/market_data_and_curves.ipynb`, `02_pricing/pricing_fundamentals.ipynb`, and `05_portfolio/portfolio_construction_and_valuation.ipynb` before this notebook. You should be comfortable with `MarketContext` JSON, instrument/portfolio specs as JSON, and the idea of repricing against a market snapshot.


## Concepts: scenarios and stress testing

A **scenario** is a declarative set of **operations** (curve shocks, spread moves, statement overrides, and similar) packaged as a typed **`ScenarioSpec`**. **Stress testing** means applying those operations to a market (and optionally a financial model), then **revaluing** positions or a portfolio so you can compare base versus stressed PVs and P&L.

finstack-quant separates typed **authoring** (templates, `ScenarioSpec`, `compose_scenarios`, parsing) from JSON-input **application** (`apply_scenario_to_market`, `apply_scenario`) and from **portfolio workflows** (`apply_scenario_and_revalue`). Call `spec.to_json()` at those JSON boundaries; the typed specs remain easy to inspect, serialize, log, and compose.


### Imports and built-in templates

`list_builtin_templates` returns template IDs from the embedded registry. Use `list_builtin_template_metadata` when you need typed `TemplateMetadata` descriptions in bulk.


In [ ]:
import json

from finstack_quant.core.dates import DayCount

from finstack_quant.scenarios import (
    apply_scenario,
    apply_scenario_to_market,
    build_from_template,
    ScenarioSpec,
    build_template_component,
    compose_scenarios,
    list_builtin_template_metadata,
    list_builtin_templates,
    list_template_components,
    validate_scenario_spec,
)

templates = list_builtin_templates()
print("Available templates:")
for t in templates:
    print(f"  {t}")
print(f"Total: {len(templates)}")
meta = list_builtin_template_metadata()
print("Template metadata entries:", len(meta))


### Build from template

`build_from_template` materializes a typed `ScenarioSpec`. Composite templates expose **components** via `list_template_components` and `build_template_component`, which also returns a `ScenarioSpec`.

Historical templates bundle rates, credit, equity, vol, and FX. For a **toy market** (only discount curves), we build a **rates component** spec for `apply_*` calls so operations reference curves that exist (`USD-SOFR` here, alongside `USD-OIS`).


In [ ]:
spec = None
scenario_for_apply = None
if templates:
    tid = templates[0]
    spec = build_from_template(tid)
    print(f"Built from {tid!r} (first 500 chars):")
    print(spec.to_json()[:500])
    components = list_template_components(tid)
    print(f"Components: {components}")
    rates_ids = [c for c in components if "rates" in c.lower()]
    if rates_ids:
        scenario_for_apply = build_template_component(tid, rates_ids[0])
        print(f"scenario_for_apply <- component {rates_ids[0]!r} (first 220 chars):")
        print(scenario_for_apply.to_json()[:220] + "...")
    elif components:
        scenario_for_apply = build_template_component(tid, components[0])
        print(f"scenario_for_apply <- first component {components[0]!r}")
        print(scenario_for_apply.to_json()[:220] + "...")
    else:
        scenario_for_apply = spec
        print("No components; using full template for apply (needs full market).")
else:
    print("No built-in templates in this build — skip template-driven cells below.")


### Custom scenario specs

Author operations with the typed **`OperationSpec`** builders — one classmethod per Rust `OperationSpec` variant (`curve_parallel_bp`, `curve_node_bp`, `market_fx_pct`, `instrument_price_pct_by_attr`, ...). Each builder validates its arguments up front, and **`.to_json()`** emits the canonical wire form, so you never have to remember field names or tag spellings.

`ScenarioSpec` wraps that operation list into a `ScenarioSpec`; `validate_scenario_spec` returns `True` when the payload parses cleanly.


In [ ]:
from finstack_quant.scenarios import CurveKind, OperationSpec


rate_shock = OperationSpec.curve_parallel_bp(CurveKind.discount(), "USD-OIS", 100.0)
print("Typed operation:  ", rate_shock)
print("Canonical wire form:", rate_shock.to_json())

custom = ScenarioSpec(
    "rate-shock-100",
    [rate_shock],
    "Rate Shock +100bp",
    "Parallel shift of USD-OIS by 100bp",
)
custom_json = custom.to_json()
print(custom_json)
valid = validate_scenario_spec(custom_json)
print(f"Valid: {valid}")
parsed = ScenarioSpec.from_json(custom_json)
print("ScenarioSpec.from_json length:", len(parsed.to_json()), "chars")


### Compose scenarios

`compose_scenarios` takes a typed **list of `ScenarioSpec` objects** and returns a single merged `ScenarioSpec`.


In [ ]:
if len(templates) >= 2:
    s1 = build_from_template(templates[0])
    s2 = build_from_template(templates[1])
    composed = compose_scenarios([s1, s2])
    print("Composed scenario (first 300 chars):", composed.to_json()[:300])
else:
    print("Need at least 2 built-in templates to demonstrate compose_scenarios (have", len(templates), ").")


### Apply scenario to market

`apply_scenario_to_market` returns an `ApplicationResult` with `.market` (`MarketContext`), `.report.operations_applied`, and `.report.warnings`.


**JSON shape vs content.** Scenario binding (`apply_scenario_to_market` / `apply_scenario`) returns a typed `ApplicationResult`; call **`.market.to_json()`** for the compact market snapshot. A notebook that prints `MarketContext.to_json()` often uses **`json.dumps(..., indent=2)`** for readability. The **curve set is the same logical data** once parsed—only whitespace and key ordering may differ.

**Extrapolation warnings.** If a scenario references **tenor pillars beyond** the curve’s built range (for example **10Y** and **30Y** nodes on a **5Y**-long curve), the engine applies the curve’s **extrapolation policy** (often **flat forward** off the last pillar). The resulting bumps can be **milder or distorted** versus a full-term structure—treat **`warnings`** as a **sanity check** that the stress is applied where you think it is.

In [ ]:
import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF
from finstack_quant.core.market_data import DiscountCurve, MarketContext

# Toy market with the same curve ids the built-in rates components shock.
# Pillars include the common template tenors so interpolant delivery stays on-curve.
knots = [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.55), (30.0, 0.22)]
as_of_d = DEMO_AS_OF
mc = MarketContext()
mc.insert(DiscountCurve("USD-OIS", as_of_d, knots, day_count="act_365f"))
mc.insert(DiscountCurve("USD-SOFR", as_of_d, knots, day_count="act_365f"))
market_json = mc.to_json()
print("Base market_json length:", len(market_json))

base_doc = json.loads(market_json)
base_curve_ids = sorted(
    c.get("id") for c in base_doc.get("curves", []) if isinstance(c, dict) and c.get("id")
)
pretty_snapshot = json.dumps(base_doc, indent=2)
pretty_ids = sorted(
    c.get("id") for c in json.loads(pretty_snapshot).get("curves", []) if isinstance(c, dict) and c.get("id")
)
print("Curve IDs: base parse == pretty parse:", base_curve_ids == pretty_ids, base_curve_ids)

if templates and scenario_for_apply:
    result = apply_scenario_to_market(scenario_for_apply.to_json(), market_json, str(as_of_d))
    print(f"Operations applied: {result.report.operations_applied}")
    print(f"Warnings: {result.report.warnings}")
    if result.report.warnings:
        print(
            "(Sanity check) Long-tenor node shocks on a short curve trigger extrapolation;"
            " compare DF behavior at the last pillar vs far tenors."
        )
    stressed_market = result.market.to_json()
    print("Stressed market_json length:", len(stressed_market))
    stressed_ids = sorted(
        c.get("id")
        for c in json.loads(stressed_market).get("curves", [])
        if isinstance(c, dict) and c.get("id")
    )
    print("Curve IDs: base vs stressed (same ids, different formatting):", base_curve_ids == stressed_ids)
    print("  base:", base_curve_ids, "| stressed:", stressed_ids)
else:
    print("Skipping apply_scenario_to_market (no template or scenario_for_apply).")


### Apply to market and statements model

`apply_scenario` mutates both **market** and **model** JSON when operations target each side. The return dict includes updated `market_json` and `model_json`.


In [ ]:
from finstack_quant.statements import ModelBuilder

b = ModelBuilder("stress-model")
b.periods("2025Q1..Q2", None)
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0)])
b.compute("gross_profit", "revenue - cogs")
model_json = b.build().to_json()
print("Model JSON length:", len(model_json))

if templates and scenario_for_apply:
    full_result = apply_scenario(scenario_for_apply.to_json(), market_json, model_json, "2025-01-15")
    print(f"Operations applied: {full_result.report.operations_applied}")
    print(f"Warnings: {full_result.report.warnings}")
    print("Market type:", type(full_result.market).__name__, "model:", type(full_result.model).__name__)
else:
    print("Skipping apply_scenario (no template or scenario_for_apply).")


## Mini-example: compose stress, revalue portfolio

Build a tiny **portfolio** JSON with one USD deposit discounted off **`USD-SOFR`** (the id shocked by built-in rates components), value it against the **base** market, then apply the **rates component** scenario and revalue via `apply_scenario_and_revalue`. Compare totals with `portfolio_result_total_value`.


In [ ]:
from finstack_quant.portfolio import PortfolioValuation, apply_scenario_and_revalue, value_portfolio

portfolio = json.dumps(
    {
        "id": "test-port",
        "as_of": str(as_of_d),
        "base_currency": "USD",
        "entities": {"E1": {"id": "E1"}},
        "positions": [
            {
                "position_id": "P1",
                "entity_id": "E1",
                "instrument_id": "DEP-1",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-1",
                        "notional": {"amount": "1000000", "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-07-15",
                        "day_count": "act_360",
                        "quote_rate": "0.05",
                        "discount_curve_id": "USD-SOFR",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            }
        ],
    }
)

# `value_portfolio` returns a typed `PortfolioValuation`; read
# `total_value` rather than digging into the raw `total_base_currency` object.
base_val = value_portfolio(portfolio, market_json)
base_total = base_val.total_value
print(f"Base portfolio total: {base_total:,.2f}")

if templates and scenario_for_apply:
    stressed_val, report = apply_scenario_and_revalue(
        portfolio, scenario_for_apply.to_json(), market_json
    )
    stressed_total = stressed_val.total_value
    dpl = stressed_total - base_total
    print(f"Stressed total:       {stressed_total:,.2f}")
    print(f"P&L impact:           {dpl:,.2f}")
    print(
        "(Sanity check) Large |P&L| here is expected when parallel + key-rate shocks hit a short SOFR curve"
        " with extrapolation; deposit PV sign follows cash-flow convention (see portfolio construction notebook)."
    )
    if report.warnings:
        print("(Note) Scenario warnings (e.g. extrapolation) also apply to this repricing path:", report.warnings)
    print("Scenario report (first 400 chars):", report.to_json()[:400])
else:
    print("Skipping stressed path (no template or scenario_for_apply).")


## Raw JSON operations (the wire format)

Operations are plain JSON objects tagged by **`kind`**, so a scenario can also live in a config file, arrive over the wire, or be generated by another system. Parse the **whole scenario JSON** with `ScenarioSpec.from_json` to obtain a typed `ScenarioSpec`.

Prefer the typed builders when authoring in Python: a misspelled field or tag is caught at construction time, whereas a hand-written dict only fails when the engine parses it — and an unrecognised key can be silently dropped. Both paths produce the **same** canonical operation, as the check below shows.


In [ ]:
raw_spec = ScenarioSpec.from_json(
    json.dumps(
        {
            "id": "raw-json-demo",
            "name": "Raw JSON Demo",
            "operations": [
                {
                    "kind": "curve_parallel_bp",
                    "curve_kind": "discount",
                    "curve_id": "USD-OIS",
                    "bp": 35.0,
                }
            ],
        }
    )
)
raw_spec_obj = json.loads(raw_spec.to_json())
print("scenario spec id:", raw_spec_obj["id"])
print("operations:      ", raw_spec_obj["operations"])

# The typed builder emits exactly the same operation.
typed_op = OperationSpec.curve_parallel_bp(CurveKind.discount(), "USD-OIS", 35.0)
print("typed == raw:", json.loads(typed_op.to_json()) == raw_spec_obj["operations"][0])


## Scenario enums & rate bindings

Scenario authoring uses typed enums — `Compounding`, `TenorMatchMode`, `TimeRollMode` — and `RateBindingSpec` binds a model node to a curve tenor under a chosen compounding/day-count.

In [ ]:
from finstack_quant.scenarios import (
    Compounding,
    TenorMatchMode,
    TimeRollMode,
    RateBindingSpec,
)

print("Compounding:", Compounding.simple(), Compounding.annual(), Compounding.semi_annual(), Compounding.continuous())
print("TenorMatchMode:", TenorMatchMode.exact(), TenorMatchMode.interpolate())
print("TimeRollMode:", TimeRollMode.calendar_days(), TimeRollMode.business_days(), TimeRollMode.approximate())

binding = RateBindingSpec(
    node_id="discount_rate",
    curve_id="USD-OIS",
    tenor="5Y",
    compounding=Compounding.annual(),
    day_count=DayCount.ACT_365F,
)
print("RateBindingSpec:", binding.to_json())

## Takeaways

- **Templates** (`list_builtin_templates`, `build_from_template`) are the fastest way to obtain realistic typed scenarios; **components** let you inspect or rebuild pieces of composite templates.
- **Custom specs** are authored with the typed **`OperationSpec`** builders (`curve_parallel_bp`, `curve_node_bp`, ...) plus `ScenarioSpec`; **`ScenarioSpec.from_json`** turns whole JSON scenario payloads into typed specs. Hand-written JSON operation dicts remain valid on the wire but are the interop path, not the authoring path.
- **`compose_scenarios`** merges a `list[ScenarioSpec]` for stacked stresses (order follows engine rules and priorities).
- **`apply_scenario_to_market`** and **`apply_scenario`** remain JSON-input boundaries, so pass `spec.to_json()`; their results provide updated typed snapshots for downstream pricing or statements.
- **`value_portfolio`** returns a typed **`PortfolioValuation`** with a `total_value` base-currency accessor.
- **`apply_scenario_and_revalue`** takes scenario JSON and returns a stressed **`PortfolioValuation`** plus a typed report — compare base vs stressed totals for P&L impact.

**Next:** combine scenario sets with reporting in `09_reporting/reporting_scenario_tearsheet.ipynb`, serializing typed specs only at JSON API boundaries.


## Analyst program: preview, compose and reprice the base book

The shock is a parallel change to USD discount zeros only. Forward and hazard curves remain
fixed, so this is not an all-rates or market-quote stress. Positive-bump sensitivities use USD
per bp. A Taylor approximation uses matched curve-space central repricing; it is not a
substitute for full repricing, and its error can have either sign.

The stressed market preserves fixed hazards. Compute PV only on that conditional market;
quote-space risk requires recalibrating the credit quotes under the changed discount curve.
The convenience scenario repricer requests risk by default, so the explicit apply-then-value
path below makes this distinction visible and avoids an inconsistent hazard replay.


In [ ]:
import json
import math
from _shared.analyst_book import AS_OF, build_book, build_market
from finstack_quant.portfolio import value_portfolio
from finstack_quant.scenarios import ScenarioSpec, OperationSpec, CurveKind, compose_scenarios, validate_scenario_spec, apply_scenario_to_market
book, market = build_book("base"), build_market("base")
unchanged = market.to_json()
base = value_portfolio(book, market, metrics=[]).total_value
def rate_scenario(bp):
    return ScenarioSpec("usd-discount-shock", [OperationSpec.curve_parallel_bp(CurveKind.discount(),"USD-OIS",bp)], f"USD discount +{bp}bp")
def reprice(bp):
    applied = apply_scenario_to_market(rate_scenario(bp), market, AS_OF)
    result = value_portfolio(book, applied.market, strict_risk=True, metrics=[])
    report = applied.report
    assert report.operations_applied == 1 and not report.warnings
    assert not json.loads(result.to_json()).get("degraded_positions",[])
    return result.total_value
scenario = rate_scenario(100)
assert validate_scenario_spec(scenario.to_json()) is None
print("Previewed operations:",json.loads(scenario.to_json())["operations"])
composed = compose_scenarios([rate_scenario(40),rate_scenario(60)])
compound_market = apply_scenario_to_market(composed, market, AS_OF)
compound_value = value_portfolio(book, compound_market.market, strict_risk=True, metrics=[])
assert abs(compound_value.total_value-reprice(100))<1e-6, "USD: additive zero shifts commute"
up, down = reprice(1), reprice(-1)
signed_dv01 = (up-down)/2
curvature_per_bp2 = up-2*base+down
rows=[]
for bp in [1,5,25,100]:
    exact = reprice(bp)-base
    taylor = signed_dv01*bp+0.5*curvature_per_bp2*bp**2
    rows.append({"bump_bp":bp,"full_pnl_usd":exact,"taylor_pnl_usd":taylor,"residual_usd":exact-taylor})
assert all(math.isfinite(row["full_pnl_usd"]) for row in rows)
small=next(row for row in rows if row["bump_bp"]==5)
large=next(row for row in rows if row["bump_bp"]==100)
# Independent 5 bp repricing: Taylor approximation within one USD cent in this fixture.
assert abs(small["residual_usd"])<.01
# Higher-order error is material at 100 bp for this same fixed-hazard experiment.
assert abs(large["residual_usd"])>1. and abs(large["residual_usd"])>abs(small["residual_usd"])
assert market.to_json()==unchanged
print("Matched scenario P&L:",rows)


## Quote-space composite, template preview, historical replay and horizon return


In [ ]:
from datetime import date,timedelta
import pandas as pd
from _shared.analyst_book import calibration_envelope, instruments, book_spec
from _shared.analyst_history import history_market
from finstack_quant.calibration import calibrate
from finstack_quant.portfolio import Portfolio, replay_portfolio
from finstack_quant.scenarios import build_from_template, list_builtin_templates, compute_horizon_return
# Quote-space stress rebuilds discount and hazard curves together. It is not an exact parallel zero shift.
quoted=calibration_envelope("base")
quote_changes=[]
for quote in quoted["market_data"]:
 if quote["kind"]=="rate_quote":
  quote["rate"]+=.01;quote_changes.append({"kind":"OIS quote","shock":"+100bp"})
 elif quote["kind"]=="cds_quote":
  quote["spread_bp"]+=50;quote_changes.append({"kind":"CDS quote","shock":"+50bp"})
print("Quote-space operation preview:",quote_changes)
calibrated=calibrate(quoted).market
composite_market=build_market("base")
composite_market.insert(calibrated.get_discount("USD-OIS"));composite_market.insert(calibrated.get_hazard("ACME-HZD"))
equity_shock=ScenarioSpec("equity-down",[OperationSpec.equity_price_pct(["SPX-SPOT"],-15)])
applied=apply_scenario_to_market(equity_shock,composite_market,AS_OF)
assert not applied.report.warnings
composite_value=value_portfolio(book,applied.market,metrics=[],strict_risk=True)
assert not json.loads(composite_value.to_json()).get("degraded_positions",[])
print({"base_usd":base,"quoted_composite_usd":composite_value.total_value,"pnl_usd":composite_value.total_value-base})
print("Available templates:",list_builtin_templates())
template=build_from_template("rate_shock_2022")
assert validate_scenario_spec(template.to_json()) is None
print("Template preview (not a historical data observation):",template.to_json())
# Dated synthetic market history; constant holdings and no intervening payments/flows/trades.
replay_dates=[date(2025,1,16)+timedelta(days=7*k) for k in range(4)]
snapshots=[(d,history_market(d)) for d in replay_dates]
replay=replay_portfolio(book,snapshots,config={"mode":"pv_and_pnl","valuation_options":{"metrics":{"mode":"only","metrics":[]}}})
assert not replay.skipped_dates
print(replay.to_dataframe().to_string(index=False))
summary=replay.summary
assert abs(float(summary["end_value"]["amount"])-float(summary["start_value"]["amount"])-float(summary["total_pnl"]["amount"]))<1e-6
horizon=ScenarioSpec("one-month-hold",[OperationSpec.time_roll_forward("1M",roll_mode="calendar_days"),OperationSpec.curve_parallel_bp("discount","USD-OIS",25)])
horizon_result=compute_horizon_return(json.dumps(instruments("base")["USD-CORP"]),history_market(replay_dates[0]),replay_dates[0],horizon,method="waterfall")
assert math.isfinite(horizon_result.total_return) and horizon_result.horizon_days>0
assert horizon_result.attribution.residual_within_tolerance()
print({"horizon_days":horizon_result.horizon_days,"total_return":horizon_result.total_return,"annualized_return":horizon_result.annualized_return,"carry_return":horizon_result.factor_contribution("carry")})


## Required exercise: design and reprice a2s10s steepener


In [ ]:
from finstack_quant.scenarios import TenorMatchMode
shape=ScenarioSpec("2s10s-steepener",[OperationSpec.curve_node_bp("discount","USD-OIS",[("2Y",-25.0),("10Y",25.0)],TenorMatchMode.interpolate())])
print("Preview:",shape.to_json())
shocked=apply_scenario_to_market(shape,market,AS_OF)
assert not shocked.report.warnings
v0=json.loads(value_portfolio(book,market,metrics=[]).to_json())
v1=json.loads(value_portfolio(book,shocked.market,metrics=[]).to_json())
rank=[]
for pid,p in v0["position_values"].items():
 rank.append({"position":pid,"pnl_usd":float(v1["position_values"][pid]["value_base"]["amount"])-float(p["value_base"]["amount"])})
rank=pd.DataFrame(rank).sort_values("pnl_usd",ascending=False)
assert abs(rank.pnl_usd.sum()-(float(v1["total_base_currency"]["amount"])-float(v0["total_base_currency"]["amount"])))<.01
print(rank.to_string(index=False));print("Largest winner for this exact curve-shape/holding convention:",rank.iloc[0].position)


## Required exercise: same shock before and after the credit extension


In [ ]:
from _shared import analyst_tracks as tracks
from finstack_quant.valuations.instruments import structured_credit_tranche_metrics
# Same100bp zero-rate operation on both books; hazards/other market factors held fixed.
credit_book=tracks.build_book("credit");credit_market=tracks.build_market("credit")
credit_shocked=apply_scenario_to_market(rate_scenario(100),credit_market,AS_OF).market
native0=value_portfolio(credit_book,credit_market,metrics=[]).total_value
native1=value_portfolio(credit_book,credit_shocked,metrics=[]).total_value
holding=tracks.clo_bbb_holding()
bbb0=structured_credit_tranche_metrics(json.dumps(holding["deal"]),holding["tranche_id"],credit_market,AS_OF).pv*holding["quantity"]
bbb1=structured_credit_tranche_metrics(json.dumps(holding["deal"]),holding["tranche_id"],credit_shocked,AS_OF).pv*holding["quantity"]
comparison=pd.DataFrame([{"book":"base","native_before_usd":base,"native_after_usd":reprice(100),"BBB_before_usd":0.0,"BBB_after_usd":0.0},{"book":"credit extended","native_before_usd":native0,"native_after_usd":native1,"BBB_before_usd":bbb0,"BBB_after_usd":bbb1}])
comparison["combined_pnl_usd"]=comparison.native_after_usd-comparison.native_before_usd+comparison.BBB_after_usd-comparison.BBB_before_usd
assert bbb0>0 and bbb1>0
assert abs(comparison.iloc[1].combined_pnl_usd-((native1+bbb1)-(native0+bbb0)))<1e-6
assert "ANALYST-CLO" not in {p["instrument_id"] for p in tracks.book_spec("credit")["positions"]}
print(comparison.to_string(index=False))
print("Native12position subtotal plus separatelyrepricedBBB. Same operation; different holdings. No universal P&L sign is assumed.")
